<a href="https://colab.research.google.com/github/dvalverdes/prueba_tecnica_DanielValverdeSalazar/blob/main/bloque0_auditoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Completitud

Pregunta:
¿Qué porcentaje de transacciones no tiene `customer_id`?
¿Es consistente con `loyalty_card = FALSE`?

Objetivo:
Evaluar si la ausencia de `customer_id` corresponde a la regla de negocio esperada.

In [ ]:
import pandas as pd
import zipfile
import os

# Define the path for the datasets directory
datasets_path = "/content/datasets"

# Create the directory if it doesn't exist
os.makedirs(datasets_path, exist_ok=True)

# Path to the zip file
zip_file_path = "/content/Datasets.zip"

# Unzip the file if it exists
if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(datasets_path)
    print(f"'{zip_file_path}' unzipped to '{datasets_path}'")
else:
    print(f"Warning: '{zip_file_path}' not found. Please ensure the dataset zip is uploaded.")

transactions = pd.read_csv(os.path.join(datasets_path, "transactions.csv"), na_values=["", "NULL", "null"])
transaction_items = pd.read_csv(os.path.join(datasets_path, "transaction_items.csv"), na_values=["", "NULL", "null"])
stores = pd.read_csv(os.path.join(datasets_path, "stores.csv"), na_values=["", "NULL", "null"])
products = pd.read_csv(os.path.join(datasets_path, "products.csv"), na_values=["", "NULL", "null"])
vendors = pd.read_csv(os.path.join(datasets_path, "vendors.csv"), na_values=["", "NULL", "null"])
store_promotions = pd.read_csv(os.path.join(datasets_path, "store_promotions.csv"), na_values=["", "NULL", "null"])

print("transactions:", transactions.shape)
print("transaction_items:", transaction_items.shape)
print("stores:", stores.shape)
print("products:", products.shape)
print("vendors:", vendors.shape)
print("store_promotions:", store_promotions.shape)

'/content/Datasets.zip' unzipped to '/content/datasets'
transactions: (174880, 8)
transaction_items: (542015, 6)
stores: (40, 8)
products: (200, 7)
vendors: (30, 5)
store_promotions: (42, 6)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
total_transacciones = len(transactions)
sin_customer_id = transactions["customer_id"].isna().sum()
porcentaje_sin_customer_id = (sin_customer_id / total_transacciones) * 100

print("Total de transacciones:", total_transacciones)
print("Transacciones sin customer_id:", sin_customer_id)
print("Porcentaje sin customer_id:", round(porcentaje_sin_customer_id, 2), "%")

Total de transacciones: 174880
Transacciones sin customer_id: 104632
Porcentaje sin customer_id: 59.83 %


In [ ]:
consistentes = sin_customer_id_df[sin_customer_id_df["loyalty_card"] == False]
inconsistentes = sin_customer_id_df[sin_customer_id_df["loyalty_card"] == True]
nulos_loyalty = sin_customer_id_df[sin_customer_id_df["loyalty_card"].isna()]

print("Sin customer_id y loyalty_card = FALSE:", len(consistentes))
print("Sin customer_id y loyalty_card = TRUE:", len(inconsistentes))
print("Sin customer_id y loyalty_card nulo:", len(nulos_loyalty))

Sin customer_id y loyalty_card = FALSE: 104632
Sin customer_id y loyalty_card = TRUE: 0
Sin customer_id y loyalty_card nulo: 0


In [ ]:
print("Porcentaje sin customer_id sobre total:", round((sin_customer_id / total_transacciones) * 100, 2), "%")
print("Porcentaje consistentes dentro de los sin customer_id:", round((len(consistentes) / sin_customer_id) * 100, 2), "%")
print("Porcentaje inconsistentes dentro de los sin customer_id:", round((len(inconsistentes) / sin_customer_id) * 100, 2), "%")

Porcentaje sin customer_id sobre total: 59.83 %
Porcentaje consistentes dentro de los sin customer_id: 100.0 %
Porcentaje inconsistentes dentro de los sin customer_id: 0.0 %


### Hallazgo de Completitud

Se identificó que el 59.83% de las transacciones no tiene `customer_id`.
El 100% de esos casos es consistente con `loyalty_card = FALSE`, por lo que no se considera una inconsistencia de calidad, sino una ausencia esperada según la regla de negocio.

## 2. Consistencia

Pregunta:
¿El `total_amount` en `transactions` coincide con la suma de `unit_price * quantity`
en `transaction_items`?

Objetivo:
Validar si el monto total reportado a nivel transacción coincide con el detalle de items.

In [ ]:
transaction_items["items_total"] = transaction_items["unit_price"] * transaction_items["quantity"]

items_resumen = transaction_items.groupby("transaction_id", as_index=False)["items_total"].sum()

consistencia = transactions.merge(items_resumen, on="transaction_id", how="left")

consistencia["difference"] = consistencia["total_amount"] - consistencia["items_total"]

consistencia.head()

,transaction_id,customer_id,transaction_date,store_id,total_amount,payment_method,loyalty_card,status,items_total,difference
0,TX_00000001,NaN,2024-01-01,TIENDA_001,120.98,CARD,False,COMPLETED,120.98,0.000000e+00
1,TX_00000002,NaN,2024-01-01,TIENDA_001,47.09,CASH,False,COMPLETED,47.09,0.000000e+00
2,TX_00000003,NaN,2024-01-01,TIENDA_001,195.55,CASH,False,COMPLETED,195.55,2.842171e-14
3,TX_00000004,CUST_01486,2024-01-01,TIENDA_001,72.25,CASH,True,COMPLETED,72.25,0.000000e+00
4,TX_00000005,NaN,2024-01-01,TIENDA_001,124.80,CASH,False,COMPLETED,124.80,0.000000e+00


In [ ]:
tolerancia = 0.01

sin_detalle = consistencia["items_total"].isna().sum()
coinciden = (consistencia["difference"].abs() <= tolerancia).sum()
no_coinciden = (consistencia["difference"].abs() > tolerancia).sum()

print("Transacciones sin detalle:", sin_detalle)
print("Transacciones que coinciden:", coinciden)
print("Transacciones que no coinciden:", no_coinciden)
print("Porcentaje que coincide:", round((coinciden / len(consistencia)) * 100, 2), "%")
print("Porcentaje que no coincide:", round((no_coinciden / len(consistencia)) * 100, 2), "%")
print("Diferencia máxima absoluta:", consistencia["difference"].abs().max())

Transacciones sin detalle: 0
Transacciones que coinciden: 173135
Transacciones que no coinciden: 1745
Porcentaje que coincide: 99.0 %
Porcentaje que no coincide: 1.0 %
Diferencia máxima absoluta: 202.6800000000003


In [ ]:
consistencia[consistencia["difference"].abs() > 0.01][["transaction_id", "total_amount", "items_total", "difference"]].head(10)

,transaction_id,total_amount,items_total,difference
92,TX_00000093,224.40,235.84,-11.44
148,TX_00000149,970.40,1055.85,-85.45
262,TX_00000263,48.99,54.45,-5.46
367,TX_00000368,30.38,33.04,-2.66
463,TX_00000464,377.55,390.78,-13.23
493,TX_00000494,46.27,50.98,-4.71
498,TX_00000499,644.05,680.51,-36.46
548,TX_00000549,62.50,70.34,-7.84
704,TX_00000705,207.89,226.67,-18.78
750,TX_00000751,38.99,41.88,-2.89


### Hallazgo de Consistencia

Se comparó el `total_amount` de `transactions` con la suma de `unit_price * quantity` calculada desde `transaction_items` para cada `transaction_id`.

- Transacciones sin detalle asociado: 0
- Transacciones que coinciden con tolerancia de 0.01: 173135
- Transacciones que no coinciden: 1745
- Porcentaje de coincidencia: 99%
- Diferencia máxima absoluta observada: 202.68

**Decisión:**  
Las diferencias menores o atribuibles a redondeo se documentan como alerta baja. Las transacciones con diferencias materiales o sin detalle asociado se marcarán como alerta para revisión en los siguientes bloques.

## 3. Unicidad

Pregunta:
Pregunta: ¿Existen transaction_id duplicados?

Objetivo:
Identificar transacciones con id duplicado

In [ ]:
duplicados = transactions[transactions.duplicated(subset=["transaction_id"], keep=False)]

print("Cantidad de filas con transaction_id duplicado:", len(duplicados))
print("Cantidad de transaction_id duplicados únicos:", duplicados["transaction_id"].nunique())

Cantidad de filas con transaction_id duplicado: 0
Cantidad de transaction_id duplicados únicos: 0


In [ ]:
duplicados.sort_values("transaction_id").head(10)

,transaction_id,customer_id,transaction_date,store_id,total_amount,payment_method,loyalty_card,status


### Hallazgo de Unicidad

Se evaluó la unicidad de `transaction_id` en la tabla `transactions`.

**Resultado:** No se identificaron `transaction_id` duplicados.

**Decisión:** Se asume `transaction_id` como llave única válida para los análisis posteriores.

## 4. Validez

**Pregunta:**  
¿Hay `total_amount` negativos o cero? ¿Hay `unit_price = 0` con `was_on_promo = FALSE`?

**Objetivo:**  
Validar si existen valores monetarios inválidos o sospechosos en transacciones e ítems, y determinar si representan errores de calidad o casos de negocio que deben marcarse para revisión.

In [ ]:
# Parte A: revisar total_amount negativos o cero en transactions
total_amount_invalidos = transactions[transactions["total_amount"] <= 0]

print("Cantidad de transacciones con total_amount <= 0:", len(total_amount_invalidos))

total_amount_invalidos[["transaction_id", "total_amount"]].head(10)

Cantidad de transacciones con total_amount <= 0: 3


,transaction_id,total_amount
36042,TX_00036043,0.0
65736,TX_00065737,0.0
108160,TX_00108161,0.0


In [ ]:
# Parte B: revisar unit_price = 0 con was_on_promo = FALSE en transaction_items
unit_price_invalidos = transaction_items[
    (transaction_items["unit_price"] == 0) &
    (transaction_items["was_on_promo"] == False)
]

print("Cantidad de items con unit_price = 0 y was_on_promo = FALSE:", len(unit_price_invalidos))

unit_price_invalidos[["transaction_id", "item_id", "unit_price", "quantity", "was_on_promo"]].head(10)

Cantidad de items con unit_price = 0 y was_on_promo = FALSE: 231


,transaction_id,item_id,unit_price,quantity,was_on_promo
1619,TX_00000521,ITEM_089,0.0,1,False
5280,TX_00001692,ITEM_089,0.0,4,False
7392,TX_00002389,ITEM_089,0.0,1,False
8910,TX_00002879,ITEM_089,0.0,1,False
11200,TX_00003628,ITEM_089,0.0,1,False
13198,TX_00004267,ITEM_089,0.0,1,False
15127,TX_00004870,ITEM_089,0.0,1,False
15344,TX_00004935,ITEM_089,0.0,1,False
16346,TX_00005258,ITEM_089,0.0,1,False
17900,TX_00005769,ITEM_089,0.0,1,False


### Hallazgo de Validez

Se evaluó la validez de los montos en `transactions` y de los precios unitarios en `transaction_items`.

**Hallazgo:**  
Se identificaron 3 transacciones con `total_amount <= 0`,
Se identificaron 231 items con `unit_price = 0` y `was_on_promo = FALSE`.

Estos casos son potencialmente inválidos o, como mínimo, requieren revisión de negocio para confirmar si corresponden a devoluciones, errores de carga o precios no justificados.

**Decisión:**  
Estos registros se marcarán como alerta de validez. En los bloques siguientes deberán revisarse antes de incluirse en análisis financieros o de desempeño comercial.

## 5. Integridad referencial

**Pregunta:**  
¿Hay `store_id` en `transactions` que no existan en `stores`? ¿`vendor_id` en `products` que no existan en `vendors`?

**Objetivo:**  
Validar que las llaves foráneas utilizadas en las tablas transaccionales y maestras tengan correspondencia en sus catálogos de referencia, para asegurar que los joins y análisis posteriores sean confiables.

In [ ]:
stores_huerfanos = transactions[~transactions["store_id"].isin(stores["store_id"])]
vendors_huerfanos = products[~products["vendor_id"].isin(vendors["vendor_id"])]

print("Transacciones con store_id inexistente en stores:", len(stores_huerfanos))
print("Productos con vendor_id inexistente en vendors:", len(vendors_huerfanos))

Transacciones con store_id inexistente en stores: 0
Productos con vendor_id inexistente en vendors: 5


In [ ]:
stores_huerfanos[["transaction_id", "store_id"]].head()
vendors_huerfanos[["item_id", "vendor_id"]].head()

,item_id,vendor_id
44,ITEM_045,VND_031
77,ITEM_078,VND_031
111,ITEM_112,VND_031
155,ITEM_156,VND_031
188,ITEM_189,VND_031


### Hallazgo de Integridad referencial

Se evaluó la integridad referencial entre `transactions.store_id` y `stores.store_id`, así como entre `products.vendor_id` y `vendors.vendor_id`.

**Hallazgo:**  
Se identificaron **0 transacciones** con `store_id` que no existe en la tabla `stores`, y **5 productos** con `vendor_id` que no existe en la tabla `vendors`.

Estos casos representan registros huérfanos y comprometen la confiabilidad de cruces y agregaciones basadas en tienda o proveedor.

**Decisión:**  
Estos registros se marcarán como alerta de integridad referencial. En los bloques siguientes deberán excluirse o tratarse por separado cuando el análisis dependa de joins con `stores` o `vendors`.

## 6. Frescura

**Pregunta:**  
¿Hay tiendas con gaps de días consecutivos sin transacciones? ¿Son esperables o sospechosos?

**Objetivo:**  
Evaluar si existen periodos sin transacciones dentro de la serie temporal de cada tienda, para identificar posibles faltantes de carga, interrupciones operativas o comportamientos esperables del negocio.

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

tx_por_dia = (
    transactions.groupby(["store_id", "transaction_date"])
    .size()
    .reset_index(name="tx_count")
)

gaps_resultados = []

for store_id, grupo in tx_por_dia.groupby("store_id"):
    fechas = grupo["transaction_date"].sort_values().drop_duplicates()
    fecha_min = fechas.min()
    fecha_max = fechas.max()

    rango_completo = pd.date_range(start=fecha_min, end=fecha_max, freq="D")
    fechas_faltantes = rango_completo.difference(fechas)

    gaps_resultados.append({
        "store_id": store_id,
        "fecha_min": fecha_min,
        "fecha_max": fecha_max,
        "dias_faltantes": len(fechas_faltantes),
        "tiene_gaps": len(fechas_faltantes) > 0
    })

gaps_df = pd.DataFrame(gaps_resultados)

print("Tiendas con al menos un gap:", gaps_df["tiene_gaps"].sum())
print("Total de tiendas analizadas:", len(gaps_df))

Tiendas con al menos un gap: 1
Total de tiendas analizadas: 40


In [ ]:
gaps_df[gaps_df["tiene_gaps"] == True].sort_values("dias_faltantes", ascending=False).head(10)

,store_id,fecha_min,fecha_max,dias_faltantes,tiene_gaps
11,TIENDA_012,2024-01-01,2025-06-30,7,True


In [ ]:
store_ejemplo = gaps_df[gaps_df["tiene_gaps"] == True].sort_values("dias_faltantes", ascending=False).head(1)["store_id"].iloc[0]

fechas_store = tx_por_dia[tx_por_dia["store_id"] == store_ejemplo]["transaction_date"].sort_values().drop_duplicates()
rango_store = pd.date_range(start=fechas_store.min(), end=fechas_store.max(), freq="D")
faltantes_store = rango_store.difference(fechas_store)

print("Store de ejemplo:", store_ejemplo)
print("Cantidad de días faltantes:", len(faltantes_store))
print(faltantes_store[:20])

Store de ejemplo: TIENDA_012
Cantidad de días faltantes: 7
DatetimeIndex(['2024-09-10', '2024-09-11', '2024-09-12', '2024-09-13',
               '2024-09-14', '2024-09-15', '2024-09-16'],
              dtype='datetime64[ns]', freq='D')


### Hallazgo de Frescura

Se evaluó la continuidad diaria de transacciones por tienda, comparando las fechas observadas contra un rango completo entre la primera y la última transacción de cada `store_id`.

**Hallazgo:**  
Se identifico **1 tienda** con al menos un gap de fechas sin transacciones dentro de su rango de actividad. En algunos casos, estos gaps pueden ser esperables por cierres operativos, baja actividad o calendarios propios del negocio; sin embargo, también podrían reflejar faltantes de carga o interrupciones en la captura de datos.

**Decisión:**  
Estos casos se documentarán como alerta de frescura. En los bloques siguientes, cualquier análisis temporal por tienda deberá considerar la existencia de gaps antes de interpretar caídas o ausencia de ventas como comportamiento real del negocio.

## 7. Integridad temporal

**Pregunta:**  
¿Existe alguna tienda con transacciones anteriores a su `opening_date`?

**Objetivo:**  
Validar que las transacciones de cada tienda ocurran únicamente a partir de su fecha oficial de apertura, para asegurar coherencia temporal en los análisis por tienda.

In [ ]:
stores["opening_date"] = pd.to_datetime(stores["opening_date"])
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

integridad_temporal = transactions.merge(
    stores[["store_id", "opening_date"]],
    on="store_id",
    how="left"
)

transacciones_antes_apertura = integridad_temporal[
    integridad_temporal["transaction_date"] < integridad_temporal["opening_date"]
]

print("Cantidad de transacciones anteriores al opening_date:", len(transacciones_antes_apertura))
print("Cantidad de tiendas afectadas:", transacciones_antes_apertura["store_id"].nunique())

Cantidad de transacciones anteriores al opening_date: 50
Cantidad de tiendas afectadas: 1


In [ ]:
transacciones_antes_apertura = transacciones_antes_apertura.copy()
transacciones_antes_apertura["dias_antes_apertura"] = (
    transacciones_antes_apertura["opening_date"] - transacciones_antes_apertura["transaction_date"]
).dt.days

print("Máxima cantidad de días antes de apertura:", transacciones_antes_apertura["dias_antes_apertura"].max() if len(transacciones_antes_apertura) > 0 else 0)

transacciones_antes_apertura[["transaction_id", "store_id", "transaction_date", "opening_date", "dias_antes_apertura"]].head(10)

Máxima cantidad de días antes de apertura: 17


,transaction_id,store_id,transaction_date,opening_date,dias_antes_apertura
168272,TX_00168273,TIENDA_037,2024-05-15,2024-06-01,17
168273,TX_00168274,TIENDA_037,2024-05-15,2024-06-01,17
168274,TX_00168275,TIENDA_037,2024-05-15,2024-06-01,17
168275,TX_00168276,TIENDA_037,2024-05-15,2024-06-01,17
168276,TX_00168277,TIENDA_037,2024-05-15,2024-06-01,17
168277,TX_00168278,TIENDA_037,2024-05-15,2024-06-01,17
168278,TX_00168279,TIENDA_037,2024-05-16,2024-06-01,16
168279,TX_00168280,TIENDA_037,2024-05-16,2024-06-01,16
168280,TX_00168281,TIENDA_037,2024-05-16,2024-06-01,16
168281,TX_00168282,TIENDA_037,2024-05-16,2024-06-01,16


### Hallazgo de Integridad temporal

Se evaluó la relación entre `transaction_date` en `transactions` y `opening_date` en `stores` para verificar que las ventas ocurran a partir de la fecha de apertura de cada tienda.

**Hallazgo:**  
Se identificaron **50 transacciones** con fecha anterior al `opening_date`, distribuidas en **1 tienda**. La mayor anticipación observada fue de **17 días** antes de la apertura oficial.

Estos casos representan una inconsistencia temporal, ya que una tienda no debería registrar transacciones antes de estar operativa.

**Decisión:**  
Estos registros se marcarán como alerta de integridad temporal. En los bloques siguientes deberán excluirse o revisarse por separado en cualquier análisis histórico por tienda.

## 8. A/B test

**Pregunta:**  
¿Hay tiendas asignadas simultáneamente a `CONTROL` y `TREATMENT` en `store_promotions`?

**Objetivo:**  
Validar que la asignación de variantes del experimento por tienda sea mutuamente exclusiva, para asegurar que el diseño del A/B test sea consistente y que los resultados posteriores no estén contaminados.

In [ ]:
ab_por_tienda = (
    store_promotions.groupby("store_id")["variant"]
    .nunique()
    .reset_index(name="cantidad_variantes")
)

tiendas_conflicto = ab_por_tienda[ab_por_tienda["cantidad_variantes"] > 1]

print("Cantidad de tiendas con más de una variante:", len(tiendas_conflicto))
print("Cantidad total de tiendas evaluadas:", ab_por_tienda["store_id"].nunique())

Cantidad de tiendas con más de una variante: 2
Cantidad total de tiendas evaluadas: 40


In [ ]:
store_promotions["start_date"] = pd.to_datetime(store_promotions["start_date"])
store_promotions["end_date"] = pd.to_datetime(store_promotions["end_date"])

conflictos_simultaneos = []

for store_id, grupo in store_promotions.groupby("store_id"):
    grupo = grupo.sort_values("start_date")
    filas = grupo.to_dict("records")

    for i in range(len(filas)):
        for j in range(i + 1, len(filas)):
            misma_tienda = filas[i]["store_id"] == filas[j]["store_id"]
            variante_distinta = filas[i]["variant"] != filas[j]["variant"]
            traslape = (
                filas[i]["start_date"] <= filas[j]["end_date"] and
                filas[j]["start_date"] <= filas[i]["end_date"]
            )

            if misma_tienda and variante_distinta and traslape:
                conflictos_simultaneos.append({
                    "store_id": store_id,
                    "variant_1": filas[i]["variant"],
                    "start_1": filas[i]["start_date"],
                    "end_1": filas[i]["end_date"],
                    "variant_2": filas[j]["variant"],
                    "start_2": filas[j]["start_date"],
                    "end_2": filas[j]["end_date"]
                })

conflictos_simultaneos_df = pd.DataFrame(conflictos_simultaneos)

print("Cantidad de conflictos simultáneos CONTROL/TREATMENT:", len(conflictos_simultaneos_df))
print("Cantidad de tiendas con conflicto simultáneo:", conflictos_simultaneos_df["store_id"].nunique() if len(conflictos_simultaneos_df) > 0 else 0)

Cantidad de conflictos simultáneos CONTROL/TREATMENT: 2
Cantidad de tiendas con conflicto simultáneo: 2


### Hallazgo de A/B test

Se evaluó la asignación de variantes en `store_promotions` para verificar si una misma tienda fue asignada simultáneamente a `CONTROL` y `TREATMENT`.

**Hallazgo:**  
Se identificaron **2 conflictos simultáneos** de variantes distintas en **Y tiendas**, con periodos de promoción traslapados entre `CONTROL` y `TREATMENT`.

Esto compromete la validez del experimento, ya que una tienda no debería pertenecer simultáneamente a ambos grupos durante el mismo periodo.

**Decisión:**  
Estos casos se marcarán como alerta crítica de diseño experimental. En los bloques siguientes deberán excluirse o tratarse por separado en cualquier análisis de impacto del A/B test.